Notebook for fit, predictive and generative evaluations of the sequential learning model.

# Upload human dataset

In [ ]:
file = "Data/json_wcst_dataset.npy"

In [3]:
import numpy as np

data = np.load(file, allow_pickle=True)
data[0]

{'step': 1,
 'key_cards': {1: ('red', 'triangle', 'one'),
  2: ('green', 'star', 'two'),
  3: ('yellow', 'cross', 'three'),
  4: ('blue', 'ball', 'four')},
 'stimulus': ('blue', 'cross', 'one'),
 'user_key': 1,
 'is_correct': False,
 'is_odd': False,
 'ground_rule': 2,
 'ground_key': 3,
 'subject_id': 1}

In [4]:
len(data) #tot_trials

63221

# Sequential Learning Model

In [5]:
# ------------------------------  WCST attention-shifting model  ------------------------------

# Bishara et al. (2010)

import numpy as np

# ---------------------------------------------------------------------
# Utility helpers
# ---------------------------------------------------------------------

def rule_type(card, stim):
    """
    Return the single dimension on which 'card' matches 'stimulus' –
    or None if they match on 0 or >1 dimensions.
    """
    hits = [card[i] == stim[i] for i in range(3)]
    return ["color", "form", "number"][hits.index(True)] if hits.count(True) == 1 else None


def match(stim, keycard, dim):
    """True iff 'keycard' shares dimension 'dim' (0=color,1=form,2=number) with 'stim'."""
    return stim[dim] == keycard[dim]


# ---------------------------------------------------------------------
# Core model: SL attention-shift family
# ---------------------------------------------------------------------

def wcst_sl_choice_prob(step,
                          state,
                          r=0.80,             # shift after reward (RIGHT)
                          p=0.50,             # shift after punishment (WRONG)
                          d=1.7,              # 'consistency' exponent that sharpens the choice probabilities
                          f=1.0,              # extra weighting used only when feedback is ambiguous
                          sample_choice=True,
                          update_from="human"):
    """
    Single-trial choice and belief update for the SL WCST model.

    Parameters
    ----------
    step : dict
        {
          'stimulus'   : tuple(color, form, number),
          'key_cards'  : {1:tuple, 2:tuple, 3:tuple, 4:tuple},
          'ground_key' : int  (correct pile 1-4)
        }
    state : dict
        Mutable store across trials – pass {} at the start of every participant.
    r, p : float
        Attention-shift rates after RIGHT / WRONG feedback (0–1).
    d : float
        Consistency exponent (>0).  d→∞ → almost deterministic, d=1 → linear weighting.
    f : float
        Focus parameter for ambiguous feedback.
    sample_choice : bool
        True  → stochastic draw;  False → greedy arg-max.
    update_from :
        human  → feeding back the human answers/feedbacks (for fitting and predictive case);
        model  → feeding back the model answers/feedbacks (for generative case).

    Returns
    -------
    choice     : int        (1-4 pile chosen)
    key_probs  : np.ndarray (shape (4,), choice probabilities for likelihood)
    """


    # ---  initialise attention vector a_t (uniform on 1st trial) -------------
    if "a" not in state:
        state["a"] = np.full(3, 1/3, dtype=float)
    a = state["a"]

    S   = np.asarray(step["stimulus"])
    KC  = step["key_cards"]

    # --- match matrix -------------------------------------------------
    match_mat = np.zeros((4, 3), dtype=int)
    for k in range(1, 5):
        match_mat[k-1] = (np.asarray(KC[k]) == S)  # row k indicates (with 0/1 flags) on which dimensions the stimulus matches pile k.

    # --- choice probabilities ----------------------------------------

    # for each pile take the element-wise product between the match flags and attention raised to the exponent **d**; sum over dimensions
    # normalise the four scores into a probability vector 'key_probs'

    pile_scores = (match_mat * (a ** d)).sum(axis=1)
    key_probs   = pile_scores / pile_scores.sum()

    # if 'sample_choice=True' draw from the soft-max, else take the highest-probability key.  Add 1 so keys are 1‒4, matching WCST dataset notation.

    choice = 1 + (np.random.choice(4, p=key_probs) if sample_choice
                  else key_probs.argmax())

    # -----------------------------------------------------------------
    # key driving the learning step
    # -----------------------------------------------------------------
    action_key = step["user_key"] if update_from == "human" else choice
    rewarded   = (action_key == step["ground_key"])     #'rewarded' is TRUE when that key equals the task’s ground-truth pile.


    # 'chosen_m' is a 3-element vector telling which dimensions the chosen pile shares with the stimulus. 'n_matches' counts 0, 1, 2, or 3 matches (relevant in case of ambiguous stimuli).

    chosen_m   = match_mat[action_key - 1]
    n_matches  = chosen_m.sum()

    # --- signal (feedback) vector s ---------------------------------------------

    # unambiguous feedback: if the pile matches on exactly one dimension, point all signal weight to that dimension when rewarded; otherwise point to the other two.
    # ambiguous feedback: two or three matches. Weight the candidate dimensions by current attention^f and renormalise.
    # 's' is always a length-3 vector that sums to 1.

    if n_matches == 1:
        s = chosen_m if rewarded else 1 - chosen_m

    else:
        num = chosen_m if rewarded else 1 - chosen_m
        num = num * (a ** f)
        s   = num / num.sum()

    # --- attention update --------------------------------------------

    # after a correct trial: a_(t+1) = (1–r)·a_t + r·s
    # after an error:    a_(t+1) = (1–p)·a_t + p·s

    a_new = ((1 - r) * a + r * s) if rewarded else ((1 - p) * a + p * s)
    state["a"] = a_new / a_new.sum()

    return choice, key_probs, pile_scores

# Global Fitting

This pipeline fits the model to predict human behaviour, tracking human’s learning history.

In [ ]:
import numpy as np
from scipy.optimize import minimize
from sklearn.model_selection import GroupShuffleSplit
from tqdm import tqdm

# -------------------------------------------------------------------
# 1.  Split by subject into train / test (sequential split)
# -------------------------------------------------------------------

# split so 80% of subjects in train, 20% in test
subj_ids = np.array([step["subject_id"] for step in data])
unique_sids = np.unique(subj_ids)
n_test = int(len(unique_sids) * 0.2)

test_sids = set(unique_sids[:n_test])
train_sids  = set(unique_sids[n_test:])

# partition the data
train_data = np.array([step for step in data if step["subject_id"] in train_sids])
test_data  = np.array([step for step in data if step["subject_id"] in test_sids])

print(f"Train subjects: {len(train_sids)}, "
      f"Test subjects: {len(test_sids)}")

# -------------------------------------------------------------------
# 2.  Negative log-likelihood under the SL attention-shift model
# -------------------------------------------------------------------

from functools import partial
from itertools import product

def neg_log_likelihood_sl(params, dataset):
    """
    params : (r, p, d, f)   with   0 ≤ r,p ≤ 1,   0 < d,f ≤ 5
    dataset: iterable of trial dicts
    """
    r, p, d, f = params
    nll   = 0.0
    state = {}                         # per-subject latent a-vector
    last_s = None

    for step in dataset:
        sid = step["subject_id"]
        if sid != last_s:
            state.clear()              # reset when new participant starts
            last_s = sid

        _, key_probs,_ = wcst_sl_choice_prob(
            step, state,
            r=r, p=p, d=d, f=f,
            sample_choice=False,       # greedy / predictive
            update_from="human"
        )
        human_key = step["user_key"] - 1
        p_k = key_probs[human_key]
        nll -= np.log(p_k + 1e-12)     # tot nll + tiny constant for numerical safety

    return nll

# -------------------------------------------------------------------
# 3.  Optimise on the training set: coarse grid → LBFGS refinement
# -------------------------------------------------------------------

# --- 3.1  coarse grid search --------------------------------------

r_grid = np.linspace(0.05, 0.95, 9)        # 0.05 … 0.95
p_grid = np.linspace(0.05, 0.95, 9)        # same
d_grid = np.geomspace(0.5, 5.0, 7)         # 0.5 … 5  (log-spaced)
f_grid = np.geomspace(0.5, 5.0, 5)         # 0.5 … 5  (log-spaced)

best = {"nll": np.inf, "params": None}

for r, p, d, f in tqdm(product(r_grid, p_grid, d_grid, f_grid),
                       total=len(r_grid)*len(p_grid)*len(d_grid)*len(f_grid),
                       desc="grid"):
    nll = neg_log_likelihood_sl((r, p, d, f), train_data)
    if nll < best["nll"]:
        best["nll"], best["params"] = nll, (r, p, d, f)

print(f"grid → r={best['params'][0]:.2f}, p={best['params'][1]:.2f}, "
      f"d={best['params'][2]:.2f}, f={best['params'][3]:.2f},  NLL={best['nll']:.1f}")


# --- 3.2  local refinement with L-BFGS-B --------------------------

x0 = np.array(best["params"])
bounds = [(1e-3, 0.999),   # r
          (1e-3, 0.999),   # p
          (0.05,  5.0),    # d
          (0.05,  5.0)]    # f

res = minimize(
    fun     = neg_log_likelihood_sl,
    x0      = x0,
    args    = (train_data,),
    bounds  = bounds,
    method  = "L-BFGS-B",
    options = {"maxiter": 400}
)

r_opt, p_opt, d_opt, f_opt = res.x
print("converged:", res.success, "|", res.message)
print(f"refined → r={r_opt:.3f}, p={p_opt:.3f}, d={d_opt:.2f}, f={f_opt:.2f}")

# -------------------------------------------------------------------
# 4.  Evaluate held-out likelihood
# -------------------------------------------------------------------

train_nll = neg_log_likelihood_sl((r_opt, p_opt, d_opt, f_opt), train_data)
test_nll  = neg_log_likelihood_sl((r_opt, p_opt, d_opt, f_opt), test_data)

n_trials_train, n_trials_test = len(train_data), len(test_data)
print(f"Train LL/trial: {-train_nll/n_trials_train:.4f}")    # grand mean
print(f" Test LL/trial: {-test_nll /n_trials_test :.4f}")    # grand mean

print(f"Mean P(correct) → train {np.exp(-train_nll/n_trials_train):.3f}, "
      f"test {np.exp(-test_nll/n_trials_test):.3f}")


Train subjects: 300, Test subjects: 75


grid: 100%|██████████| 2835/2835 [2:16:31<00:00,  2.89s/it]


grid → r=0.95, p=0.61, d=0.50, f=0.50,  NLL=44733.0
converged: True | CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH
refined → r=0.967, p=0.656, d=0.41, f=0.05
Train LL/trial: -0.8796
 Test LL/trial: -0.8711
Mean P(correct) → train 0.415, test 0.418


# Generative Case

### Simulation with first 75 subjects (test set)

In [6]:
# extract arrays of subject IDs
subj_ids = np.array([step["subject_id"] for step in data])
unique_sids = np.unique(subj_ids)
# we'll split so 80% of subjects in train, 20% in test
n_test = int(len(unique_sids) * 0.2)

test_sids = set(unique_sids[:n_test])
train_sids  = set(unique_sids[n_test:])

# Partition the data
train_data = np.array([step for step in data if step["subject_id"] in train_sids])
test_data  = np.array([step for step in data if step["subject_id"] in test_sids])

print(f"Train subjects: {len(train_sids)}, "
      f"Test subjects: {len(test_sids)}")

Train subjects: 300, Test subjects: 75


In [11]:
from collections import defaultdict
import numpy as np, random, torch
from tqdm import tqdm

# ------------------------------------------------------------------
# 0.  Parameters of the fitted SL model
# ------------------------------------------------------------------
r_fit, p_fit, d_fit, f_fit = 0.967, 0.656, 0.41, 0.05

# ------------------------------------------------------------------
# 1.  Dictionaries and variables
# ------------------------------------------------------------------
choices           = []
human_correct     = defaultdict(list)
model_correct     = defaultdict(list)
model_aligned     = defaultdict(list)

model_persev      = defaultdict(list)
model_setloss     = defaultdict(list)
human_persev      = defaultdict(list)
human_setloss     = defaultdict(list)

current_id        = None
model_state       = {}

prev_m_rule = prev_h_rule = None
prev_m_ok   = prev_h_ok   = None

# ------------------------------------------------------------------
# 2.  Generative simulation on the test set
# ------------------------------------------------------------------
for step in tqdm(test_data):

    # -- handle new subject (deterministic seeds + state reset) ----
    if current_id != step['subject_id']:
        current_id  = step['subject_id']
        seed        = 10_000 + current_id
        random.seed(seed);  np.random.seed(seed)
        torch.manual_seed(seed);  torch.cuda.manual_seed_all(seed)
        model_state = {}             # reset latent attention

        prev_m_rule = prev_h_rule = None
        prev_m_ok   = prev_h_ok   = None

    # -- human correctness ------------------------------------------
    h_ok = int(step['user_key'] == step['ground_key'])
    human_correct[current_id].append(h_ok)

    # -- model choice (sampling) and statistics ---------------------
    m_key, k_probs, _ = wcst_sl_choice_prob(
        step, model_state,
        r=r_fit, p=p_fit, d=d_fit, f=f_fit,
        sample_choice=True,           # draw from soft-max (generative)
        update_from="model"           # learn from the model’s own key
    )

    choices.append(m_key)
    m_ok = int(m_key == step['ground_key'])
    model_correct[current_id].append(m_ok)
    model_aligned[current_id].append(int(m_key == step['user_key']))

    #   matching rule
    h_rule = rule_type(step['key_cards'][step['user_key']], stim=step['stimulus'])
    m_rule = rule_type(step['key_cards'][m_key], stim=step['stimulus'])

    #   perseveration: two consecutive errors with same wrong rule
    model_persev[current_id].append(int(prev_m_ok == 0 and m_ok == 0 and m_rule == prev_m_rule))
    human_persev[current_id].append(int(prev_h_ok == 0 and h_ok == 0 and h_rule == prev_h_rule))

    #   set-loss: correct → error with a different rule
    model_setloss[current_id].append(int(prev_m_ok == 1 and m_ok == 0 and m_rule != prev_m_rule))
    human_setloss[current_id].append(int(prev_h_ok == 1 and h_ok == 0 and h_rule != prev_h_rule))

    prev_m_rule, prev_h_rule, prev_m_ok, prev_h_ok = m_rule, h_rule, m_ok, h_ok

# ------------------------------------------------------------------
# 3.  Summary metrics
# ------------------------------------------------------------------
avg_acc       = np.mean([np.mean(model_correct[s]) for s in model_correct])
avg_alignment = np.mean([np.mean(model_aligned[s]) for s in model_aligned])
avg_persev    = np.mean([np.mean(model_persev[s])  for s in model_persev])
avg_setloss   = np.mean([np.mean(model_setloss[s]) for s in model_setloss])


print(f"r={r_fit:.3f} p={p_fit:.3f} d={d_fit:.2f} f={f_fit:.2f}")

print(f"Accuracy (model vs ground key):     {avg_acc:.3f}")
print(f"Alignment (model vs human choice):  {avg_alignment:.3f}")
print(f"Perseveration:                      {avg_persev:.3f}")
print(f"Set-loss:                           {avg_setloss:.3f}")

100%|██████████| 12754/12754 [00:01<00:00, 10865.80it/s]

r=0.967 p=0.656 d=0.41 f=0.05
Accuracy (model vs ground key):     0.489
Alignment (model vs human choice):  0.543
Perseveration:                      0.095
Set-loss:                           0.047


In [9]:
np.save('SL_model_correct_generative.npy', model_correct)
np.save('SL_model_aligned_generative.npy', model_aligned)
np.save('SL_model_persev_generative.npy', model_persev)
np.save('SL_model_setloss_generative.npy', model_setloss)

# Predictive Case

### Simulation with first 75 subjects (test set)

In [16]:
# ------------------------------------------------------------------
# 0.  Fitted parameters
# ------------------------------------------------------------------
r_fit, p_fit, d_fit, f_fit = 0.967, 0.656, 0.41, 0.05

# ------------------------------------------------------------------
# 1.  Dictionaries and variables
# ------------------------------------------------------------------
from collections import defaultdict
import numpy as np
from tqdm import tqdm
import random, torch

human_correct  = defaultdict(list)
model_correct  = defaultdict(list)
model_aligned  = defaultdict(list)
model_logL     = defaultdict(list)

choices       = []
n_trials = len(test_data)            # tot test trials
nll = 0.0                            # accumulate nll
current_id    = None
model_state   = {}                   # subject-specific latent attention vector

# ------------------------------------------------------------------
# 2.  Main prediction loop
# ------------------------------------------------------------------
for step in tqdm(test_data):

    # ── subject switch ⇒ deterministic RNG + state reset ──
    if current_id != step['subject_id']:
        current_id  = step['subject_id']
        seed        = 10_000 + current_id
        random.seed(seed);  np.random.seed(seed)
        torch.manual_seed(seed);  torch.cuda.manual_seed_all(seed)
        model_state = {}                          # RESET attention for new subject

    # -- human correctness on this trial ----------------------------
    human_ok = int(step['user_key'] == step['ground_key'])
    human_correct[current_id].append(human_ok)

    # -- model prediction, accuracy, alignment, log-likelihood ------
    m_key, k_probs,_ = wcst_sl_choice_prob(
        step, model_state,
        r=r_fit, p=p_fit, d=d_fit, f=f_fit,
        sample_choice=False,          # greedy arg-max prediction
        update_from="human"           # update attention using the human key
    )

    choices.append(m_key)
    model_correct[current_id].append(int(m_key == step['ground_key']))
    model_aligned[current_id].append(int(m_key == step['user_key']))

    # ----- nll --------------------
    model_logL[current_id].append(np.log(k_probs[step['user_key']-1] + 1e-12))
    nll -= np.log(k_probs[step["user_key"]-1] + 1e-12)

# ------------------------------------------------------------------
# 3.  Aggregate metrics
# ------------------------------------------------------------------

# mean of means
avg_acc        = np.mean([np.mean(model_correct[s]) for s in model_correct])
avg_alignment  = np.mean([np.mean(model_aligned[s]) for s in model_aligned])
avg_ll         = np.mean([np.mean(model_logL[s]) for s in model_logL])

# ---------------------------------------------------------------
print(f"r={r_fit:.3f}  p={p_fit:.3f}  d={d_fit:.2f}  f={f_fit:.2f}")
print(f"Accuracy (model vs ground key):     {avg_acc:.3f}")
print(f"Alignment (model vs human choice):  {avg_alignment:.3f}")

print(f"lengh trials: {n_trials}")
print(f"tot NLL: {nll}")

print(f"Test avg nll:  {-avg_ll:.3f}")                              # mean of subject means
print(f"Test grand-mean (NLL/trial):  {nll / n_trials:.3f} ")         # grand mean across

100%|██████████| 12754/12754 [00:00<00:00, 17758.55it/s]

r=0.967  p=0.656  d=0.41  f=0.05
Accuracy (model vs ground key):     0.604
Alignment (model vs human choice):  0.719
lengh trials: 12754
tot NLL: 11113.399311779445
Test avg nll:  0.841
Test grand-mean (NLL/trial):  0.871 


In [15]:
np.save('SL_model_correct_predictive.npy', model_correct)
np.save('SL_model_aligned_predictive.npy', model_aligned)
np.save('SL_model_loglikelihood_predictive.npy', model_logL)